## Qwen2.5をローカルLLMとして立ち上げます

### 最初に各セルのテキストコメントを一読してください

### ●ランタイムはT4 GPUとしておいてください
### ●Google Driveにマウントします
### 注意：Google Derive の※1（/content/drive/MyDrive/models/gguf）というフォルダを自動にて作成します
### この ※1フォルダにLLMのモデルをダウンロードします。6GBくらい使用しますので、十分な空きがあるか必ずチェックしてください。空きがない場合は、Google Colabの実行環境下のフォルダに変更してください

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = "/content/drive/MyDrive/models/gguf" # Google Drive (/content/drive/MyDrive )を使用します。6GBくらい領域を消費します。

Mounted at /content/drive


Qwen/Qwen2.5-7B-Instruct-GGUF実行のためのllamaインストール

In [ ]:
!pip install -q llama-cpp-python \
  --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 551.3/551.3 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.5 MB/s eta 0:00:00


### Qwen/Qwen2.5-7B-Instruct-GGUF実行環境を整える
ダウンロードを実行するときにHugging Faceのトークンを使用していると思います。  
Hugging Faceのトークンをあらかじめ作成しておく必要があります。(ここでは、Google Colabの「シークレット」にHF_TOKENで登録しています）


In [ ]:
import os
from huggingface_hub import hf_hub_download

MODEL_ID  = "Qwen/Qwen2.5-7B-Instruct-GGUF"
os.makedirs(DRIVE_DIR, exist_ok=True)

FILENAMES = [
    "qwen2.5-7b-instruct-q4_k_m-00001-of-00002.gguf",
    "qwen2.5-7b-instruct-q4_k_m-00002-of-00002.gguf",
]

for filename in FILENAMES:
    local_path = os.path.join(DRIVE_DIR, filename)
    if os.path.exists(local_path):
        print(f"✅ スキップ: {filename}")
    else:
        print(f"⏬ ダウンロード中: {filename}")
        hf_hub_download(repo_id=MODEL_ID, filename=filename, local_dir=DRIVE_DIR)
        print(f"✅ 完了: {filename}")

✅ スキップ: qwen2.5-7b-instruct-q4_k_m-00001-of-00002.gguf
✅ スキップ: qwen2.5-7b-instruct-q4_k_m-00002-of-00002.gguf


Qwen/Qwen2.5-7B-Instruct-GGUFをロードし、実行できる状態にする

In [ ]:
import os
from llama_cpp import Llama

# 最初のファイルを指定するだけでOK
MODEL_PATH = "/content/drive/MyDrive/models/gguf/qwen2.5-7b-instruct-q4_k_m-00001-of-00002.gguf"

print("🔄 モデルをロード中...")
llm = Llama(
    model_path=MODEL_PATH,
    n_gpu_layers=-1,    # 全レイヤーをT4 GPUへ
    n_ctx=4096,
    verbose=False,
)
print("✅ ロード完了")

def chat(user_message,
         system_prompt="You are a helpful assistant.",
         max_tokens=512,
         temperature=0.7,
         top_p=0.9):
    response = llm.create_chat_completion(
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_message},
        ],
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p,
    )
    return response["choices"][0]["message"]["content"]

print(chat("日本の四季の魅力を100字程度で教えてください。"))


Fast API 実行環境構築

In [ ]:
print('install start')
!pip -q install pyngrok
!pip -q install fastapi uvicorn nest-asyncio
print('install end')

Fast API構築

In [ ]:
import nest_asyncio
nest_asyncio.apply()

from fastapi import FastAPI
from pydantic import BaseModel
import uvicorn
from pyngrok import ngrok

app = FastAPI()

# ── LLM リクエスト/レスポンスの型定義 ──────────────────

class LLMRequest(BaseModel):
    system_prompt: str="You are a helpful assistant."
    max_tokens: int=512
    temperature: float=0.7
    top_p: float=0.9

class LLMResponse(BaseModel):
    response: str = '固定のレスポンスデータ'

#POST LLMの回答を得る
# POST: JSONボディを受け取る
@app.post("/llms", response_model=LLMResponse)
def create_llm(llm: LLMRequest):
    print('llm.system_prompt=',llm.system_prompt)
    response = chat(
        user_message=llm.system_prompt)
    result = LLMResponse()
    result.response=response
    return result



### ngrokによるGWルートに FastAPIを接続
### https://ngrok.com/にアカウント登録してから、認証トークンを取得しておく必要があります
### 認証トークンは、Google Colabの「シークレット」に「NGROK_AUTHTOKEN」で登録しています
### 🚀URL:https://xxxxxxxxxxxxxxxx..ngrok-free.appをメモししておいてください。


In [ ]:
from pyngrok import ngrok
from google.colab import userdata
# https://dashboard.ngrok.com/authtokens で無料取得したトークンを設定
ngrok.set_auth_token(userdata.get("NGROK_AUTHTOKEN"))

# ── 起動 ────────────────────────────────────────────
# ngrok.set_auth_token("YOUR_NGROK_TOKEN")  # ← 自分のトークンに変更
ngrok.set_auth_token(userdata.get("NGROK_AUTHTOKEN"))
public_url = ngrok.connect(8000)
print(f"🚀 URL: {public_url}")
print(f"📖 Swagger UI: {public_url}/docs")

config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
server = uvicorn.Server(config)
await server.serve()